# IDS basado en IA — Huella digital IEEE 802.3 · Cuaderno de cierre (E7)

Recorrido **CRISP-DM** de punta a punta del proyecto (OE1–OE3). Este cuaderno **no reentrena nada**:
cada cifra se lee de los JSON congelados en `reports/`, y cada celda cita la etapa (E1–E6) y el
archivo del que sale. Si un JSON no trae una cifra que el texto afirma, la celda lo dice
(«cifra no encontrada en reports/, ver informe X») en lugar de inventarla.

Todos los datos son **tráfico sintético de laboratorio** (generadores con semilla fija de
`scripts/lab/`), no capturas de una red real.

| Fase CRISP-DM | Etapas | Fuente principal |
|---|---|---|
| 1. Comprensión del negocio | E1, E4 (OE1) | `reports/robustez_e4c.json` (invariantes), `reports/dataset_lab_2026-11-13.json` |
| 2. Comprensión de los datos | E1 | `reports/dataset_lab_2026-11-13.json` |
| 3. Preparación de los datos | E1 / Extractor 3.0 | `reports/extractor_final.json`, `reports/evaluacion_e2.json` |
| 4. Modelado | E2, Extractor, Anomalías, E3A/E3B | `reports/evaluacion_e2.json`, `reports/extractor_final.json`, `reports/anomalias.json`, `reports/identidad_e3b.json` |
| 5. Evaluación | E4R, E4C, E5C (OE3) | `reports/robustez_e4.json`, `reports/robustez_e4c.json`, `reports/adopcion_e5c.json` |
| 6. Despliegue | E5, E6 (OE3) | sin JSON en `reports/`: `docs/INFORME_E5.md`, `docs/INFORME_E6.md` |

Ejecútalo desde la raíz del repositorio o desde `notebooks/`.

In [1]:
# Preparación: localizar la raíz y definir el lector de reports/ (no entrena ni genera datos).
import json, sys, time
from pathlib import Path

import pandas as pd

T_INICIO = time.perf_counter()
RAIZ = Path.cwd() if (Path.cwd() / "reports").exists() else Path.cwd().parent
assert (RAIZ / "reports").exists(), "Ejecuta el cuaderno desde la raíz del repo o desde notebooks/."
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

VERIFICACION = []  # (fase, cifra del texto, valor en JSON, archivo, ruta, coincide)


def leer(archivo):
    return json.loads((RAIZ / "reports" / archivo).read_text(encoding="utf-8"))


def obtener(archivo, ruta, informe):
    # Devuelve el valor de reports/<archivo> en `ruta` (claves separadas por '/'), o None con aviso.
    p = RAIZ / "reports" / archivo
    try:
        d = json.loads(p.read_text(encoding="utf-8"))
        for k in ruta.split("/"):
            d = d[int(k)] if isinstance(d, list) else d[k]
        return d
    except (FileNotFoundError, KeyError, IndexError, ValueError):
        print(f"cifra no encontrada en reports/ ({archivo}:{ruta}), ver informe {informe}")
        return None


def coma(x, dec=3):
    return "—" if x is None else f"{x:.{dec}f}".replace(".", ",")


def registrar(fase, texto, valor, archivo, ruta, dec=3):
    esperado = float(texto.replace(",", "."))
    ok = valor is not None and round(float(valor), dec) == round(esperado, dec)
    VERIFICACION.append({"fase": fase, "cifra del texto": texto, "valor en JSON": valor,
                         "archivo": f"reports/{archivo}", "ruta": ruta, "coincide": ok})
    return ok


print("Raíz del repo:", RAIZ.name)
print("JSON en reports/:", len(list((RAIZ / "reports").glob("**/*.json"))))

Raíz del repo: ids_ia
JSON en reports/: 21


## 1. Comprensión del negocio — vectores de ataque por capa (OE1; E1 y E4)

El IDS defiende un segmento Ethernet IEEE 802.3. El catálogo cubre seis ataques en tres capas
TCP/IP. Para cada uno, el **invariante de efecto** (lo que el ataque necesita para seguir siendo un
ataque) fija la tasa mínima con efecto; E4 lo usa para separar «evasión» de «el ataque dejó de
serlo».

Fuente: `reports/robustez_e4c.json` → `invariantes` (E4C) y `reports/dataset_lab_2026-11-13.json` →
`por_capa` (E1).

In [2]:
inv = obtener("robustez_e4c.json", "invariantes", "docs/INFORME_E4C.md") or []
capas = obtener("dataset_lab_2026-11-13.json", "por_capa", "docs/INFORME_E1.md") or {}
CAPA = {"mac_flooding": "enlace", "arp_spoofing": "enlace", "icmp_flood": "red",
        "syn_flood": "transporte", "port_scan": "transporte", "dns_spoofing": "aplicación"}
print("Ventanas por capa en el dataset de desarrollo (E1):", capas)
pd.DataFrame([{"ataque": x["ataque"], "capa": CAPA.get(x["ataque"], "?"),
               "invariante de efecto": x["invariante"], "suelo de efecto": x["suelo"]} for x in inv])

Ventanas por capa en el dataset de desarrollo (E1): {'aplicacion': 621, 'enlace': 3399, 'red': 620, 'transporte': 1075}


,ataque,capa,invariante de efecto,suelo de efecto
0,syn_flood,transporte,agotar el backlog (N) antes del primer reinten...,N/RTO ≈ 64/3 ≈ 21 medios-abiertos/s
1,arp_spoofing,enlace,refrescar antes del gc_stale de la caché (≈60 ...,"≥ 1/60 s ≈ 0,017 ARP/s"
2,mac_flooding,enlace,inserciones/s ≥ tamaño de la tabla CAM / su ti...,≈ 8192/300 ≈ 27 MAC/s
3,port_scan,transporte,puertos distintos/hora suficientes para cubrir...,depende del rango; top-puertos se cubre lento
4,dns_spoofing,aplicación,la respuesta forjada llega antes que la legíti...,1 respuesta por consulta dentro de la ventana ...
5,icmp_flood,red,saturar el ancho de banda o la CPU del objetiv...,depende del enlace; muy por encima de unas poc...


## 2. Comprensión de los datos — dataset del laboratorio (E1)

20 sesiones sintéticas de 15 min (generador congelado `scripts/lab/generar_trafico.py`), agregadas
en ventanas de 5 s por MAC de origen. El sha256 identifica el contrato de datos de todo el proyecto.

Fuente: `reports/dataset_lab_2026-11-13.json` (ficha del dataset, E1).

In [3]:
ficha = leer("dataset_lab_2026-11-13.json")
sha = ficha["sha256"]
print(f"archivo: {ficha['archivo']}  |  esquema {ficha['version_esquema']}")
print(f"sha256:  {sha}  (abreviado {sha[:8]}…{sha[-4:]})")
print(f"filas: {ficha['filas']}  columnas: {ficha['n_columnas']}  sesiones: {ficha['sesiones']}  "
      f"proporción maliciosa: {coma(ficha['proporcion_maliciosas'])}")
assert sha.startswith("7f06d099") and sha.endswith("a945"), "El sha256 no es el del contrato 7f06d099…a945"
pd.Series(ficha["por_tipo_ataque"], name="ventanas").sort_values(ascending=False).to_frame()

archivo: data/dataset_lab.csv  |  esquema 3.0
sha256:  7f06d099477a474b12db73001f3ee56a8e14e93bb41501c8b49ee94e4032a945  (abreviado 7f06d099…a945)
filas: 78988  columnas: 65  sesiones: 20  proporción maliciosa: 0,072


,ventanas
normal,73273
mac_flooding,2943
syn_flood,653
dns_spoofing,621
icmp_flood,620
arp_spoofing,456
port_scan,422


## 3. Preparación de los datos — esquema 3.0 (E1 / Extractor)

El extractor (`src/ventanas.py`, compartido por el lote y el streaming) produce las columnas del
**esquema 3.0**; el modelo de ataques consume 52. Las de capa 2 pura se evaluaron aparte (LOAO «sin
capa 2»).

Fuente: `reports/evaluacion_e2.json` → `columnas_modelo`, `version_esquema` (E2) y
`reports/extractor_final.json` → `features_3_0` (Extractor).

In [4]:
cols = obtener("evaluacion_e2.json", "columnas_modelo", "docs/INFORME_E2.md") or []
ver = obtener("evaluacion_e2.json", "version_esquema", "docs/INFORME_E2.md")
f30 = obtener("extractor_final.json", "features_3_0", "docs/INFORME_EXTRACTOR.md")
print(f"Esquema {ver}: {len(cols)} columnas de modelo")
registrar("Preparación", "52", len(cols), "evaluacion_e2.json", "columnas_modelo (longitud)", 0)
print("Primeras columnas:", cols[:10])
print("features_3_0 (extractor_final.json):", type(f30).__name__,
      len(f30) if hasattr(f30, "__len__") else f30)

Esquema 3.0: 52 columnas de modelo
Primeras columnas: ['n_tramas', 'bytes_totales', 'longitud_trama_media', 'longitud_trama_std', 'iat_media_ms', 'iat_std_ms', 'tasa_unicast', 'tasa_multicast', 'tasa_broadcast', 'macs_destino_distintas']
features_3_0 (extractor_final.json): dict 52


## 4. Modelado

### 4.1 Clasificador supervisado por episodio (OE2; E2)

RF y XGBoost sobre las 52 columnas, validación por sesión (5 folds × 5 semillas), umbral fijado a
1 FA/disp-día. Cifra del informe: **XGB recall por episodio 0,894 [0,856, 0,936] a 0,89 FA/disp-día**.

Fuente: `reports/evaluacion_e2.json` → `modelos/<modelo>/resumen` (E2).

In [5]:
filas = []
for m in ("xgboost", "random_forest"):
    base = f"modelos/{m}/resumen"
    rec = obtener("evaluacion_e2.json", f"{base}/episodios/recall", "docs/INFORME_E2.md")
    fa = obtener("evaluacion_e2.json", f"{base}/dispositivos/falsas_alarmas_por_dispositivo_dia/valor", "docs/INFORME_E2.md")
    pr = obtener("evaluacion_e2.json", f"{base}/ventanas_referencia/pr_auc/valor", "docs/INFORME_E2.md")
    filas.append({"modelo": m, "recall/episodio": coma(rec["valor"]),
                  "IC95": f"[{coma(rec['ic95'][0])}, {coma(rec['ic95'][1])}]",
                  "n episodios": rec.get("n"), "FA/disp-día": coma(fa, 2), "PR-AUC ventana": coma(pr)})
r = "modelos/xgboost/resumen/episodios/recall"
registrar("Modelado E2", "0,894", obtener("evaluacion_e2.json", f"{r}/valor", "-"), "evaluacion_e2.json", f"{r}/valor")
registrar("Modelado E2", "0,856", obtener("evaluacion_e2.json", f"{r}/ic95/0", "-"), "evaluacion_e2.json", f"{r}/ic95/0")
registrar("Modelado E2", "0,936", obtener("evaluacion_e2.json", f"{r}/ic95/1", "-"), "evaluacion_e2.json", f"{r}/ic95/1")
r = "modelos/xgboost/resumen/dispositivos/falsas_alarmas_por_dispositivo_dia/valor"
registrar("Modelado E2", "0,89", obtener("evaluacion_e2.json", r, "-"), "evaluacion_e2.json", r, 2)
pd.DataFrame(filas)

,modelo,recall/episodio,IC95,n episodios,FA/disp-día,PR-AUC ventana
0,xgboost,"0,894","[0,856, 0,936]",188,"0,89","0,989"
1,random_forest,"0,878","[0,834, 0,922]",188,"1,19","0,986"


### 4.2 Transferencia a tipos nuevos (Extractor, LOAO) y detector solo-benigno (Anomalías)

*Leave-one-attack-out* sin capa 2: el supervisado no transfiere a ataques que no vio
(**XGB 0,228 [0,168, 0,286]**). Un detector entrenado solo con benigno tampoco lo arregla (IF 0,185).

Fuentes: `reports/extractor_final.json` → `vigente` (Extractor) y `reports/anomalias.json` →
`resultados/isolation_forest/sin_capa2/resumen` (Anomalías).

In [6]:
loao = obtener("extractor_final.json", "vigente", "docs/INFORME_EXTRACTOR.md")
for m in ("xgboost", "random_forest"):
    v = loao[m]["media"]
    print(f"LOAO sin capa 2 · {m:13s}: {coma(v['valor'])} [{coma(v['ic95'][0])}, {coma(v['ic95'][1])}]")
registrar("Modelado LOAO", "0,228", loao["xgboost"]["media"]["valor"], "extractor_final.json", "vigente/xgboost/media/valor")
registrar("Modelado LOAO", "0,168", loao["xgboost"]["media"]["ic95"][0], "extractor_final.json", "vigente/xgboost/media/ic95/0")
registrar("Modelado LOAO", "0,286", loao["xgboost"]["media"]["ic95"][1], "extractor_final.json", "vigente/xgboost/media/ic95/1")
# La clave del detector contiene «/» («isolation_forest/sin_capa2»): se lee sin el separador de ruta.
an = obtener("anomalias.json", "resultados", "docs/INFORME_ANOMALIAS.md") or {}
r = "resultados/«isolation_forest/sin_capa2»/resumen/recall_episodios/valor"
iforest = an.get("isolation_forest/sin_capa2", {}).get("resumen", {}).get("recall_episodios", {}).get("valor")
if iforest is None:
    print("cifra no encontrada en reports/ (anomalias.json), ver informe docs/INFORME_ANOMALIAS.md")
print(f"Isolation Forest solo-benigno, sin capa 2: {coma(iforest)}")
registrar("Modelado Anomalías", "0,185", iforest, "anomalias.json", r)
pd.DataFrame({m: {t: v["valor"] for t, v in loao[m]["por_tipo"].items()} for m in loao}).round(3)

LOAO sin capa 2 · xgboost      : 0,228 [0,168, 0,286]
LOAO sin capa 2 · random_forest: 0,068 [0,032, 0,105]
Isolation Forest solo-benigno, sin capa 2: 0,185


,random_forest,xgboost
arp_spoofing,0.147,0.294
dns_spoofing,0.100,0.100
icmp_flood,0.097,0.161
mac_flooding,0.000,0.265
port_scan,0.000,0.321
syn_flood,0.065,0.226


### 4.3 Identidad del dispositivo: perfil de pila y vínculo IP→MAC (OE2/OE3; E3A, E3B)

Canal independiente del clasificador de ataques. Juzgar el IP-ID **por flujo** baja las falsas
alarmas de **16,19 a 0,38 FA/disp-día** sin perder detección de clones de pila distinta.

Fuente: `reports/identidad_e3b.json` → `resumen` (E3B; incluye la referencia E3A).

In [7]:
e3 = leer("identidad_e3b.json")["resumen"]
print(f"IP-ID por flujo: FA {coma(e3['ipid']['fa_antes'], 2)} → {coma(e3['ipid']['fa_despues'], 2)} FA/disp-día")
registrar("Modelado E3", "16,19", e3["ipid"]["fa_antes"], "identidad_e3b.json", "resumen/ipid/fa_antes", 2)
registrar("Modelado E3", "0,38", e3["ipid"]["fa_despues"], "identidad_e3b.json", "resumen/ipid/fa_despues", 2)
cv = e3["con_variacion"]["con_variacion"]
print(f"Con variación benigna: {coma(cv['fa_ventanas_dia'], 2)} FA/disp-día (no_confiable {coma(cv['no_confiable_ventanas'])})")
registrar("Modelado E3", "826", round(cv["fa_ventanas_dia"]), "identidad_e3b.json", "resumen/con_variacion/con_variacion/fa_ventanas_dia (redondeo)", 0)
niv = {k: v for k, v in e3["niveles"].items() if isinstance(v, dict) and "deteccion" in v}
registrar("Modelado E3", "0,20", niv["pila_completa"]["deteccion"], "identidad_e3b.json", "resumen/niveles/pila_completa/deteccion", 2)
pd.DataFrame(niv).T.rename(columns={"deteccion": "detección", "latencia_s": "latencia (s)"})

IP-ID por flujo: FA 16,19 → 0,38 FA/disp-día
Con variación benigna: 826,24 FA/disp-día (no_confiable 0,128)


,detección,latencia (s)
ingenuo,0.975,3.7
ritmo,1.000,10.0
pila_completa,0.200,8.6
suplantacion_ip,0.950,3.9
victima_apagada,1.000,4.6


## 5. Evaluación — robustez y evasión a nivel de pcap (OE3; E4R, E4C, E5C)

### 5.1 Recall por intensidad (E4R)

Fuente: `reports/robustez_e4.json` → `resumen` (E4R).

In [8]:
e4 = leer("robustez_e4.json")["resumen"]
ri = e4["recall_por_intensidad"]
for k in ("media", "suave", "rafaga"):
    print(f"recall supervisado · {k:7s}: {coma(ri[k])}")
registrar("Evaluación E4R", "0,935", ri["media"], "robustez_e4.json", "resumen/recall_por_intensidad/media")
registrar("Evaluación E4R", "0,891", ri["suave"], "robustez_e4.json", "resumen/recall_por_intensidad/suave")
registrar("Evaluación E4R", "0,824", ri["rafaga"], "robustez_e4.json", "resumen/recall_por_intensidad/rafaga")
can = e4["canales"]
print(f"invisibles por construcción: {e4['invisibles_total']}")
print(f"OR con canales de identidad: +{coma(can['extra_recall_or'])} de recall a "
      f"{coma(can['fa_por_dispositivo_dia']['identidad_en_estas_sesiones'], 2)} FA/disp-día de identidad")
_ = registrar("Evaluación E4R", "0,053", can["extra_recall_or"], "robustez_e4.json", "resumen/canales/extra_recall_or")

recall supervisado · media  : 0,935
recall supervisado · suave  : 0,891
recall supervisado · rafaga : 0,824
invisibles por construcción: 0
OR con canales de identidad: +0,053 de recall a 122,06 FA/disp-día de identidad


### 5.2 Régimen invisible del ARP lento (E4C)

Separando las tramas ARP ≥ 5 s, ninguna ventana alcanza K=2: el supervisado pasa de **1,000 a
0,000**; el **vigilante IP→MAC** lo detecta al **1,000** con **0,0 FA/disp-día** sobre variación
benigna. Ojo: son 2 episodios por paso (`n`), cifra de existencia, no de tasa.

Fuente: `reports/robustez_e4c.json` → `resumen` (E4C).

In [9]:
e4c = leer("robustez_e4c.json")["resumen"]
arp = e4c["recall_arp_por_paso"]
tabla = pd.DataFrame(arp).T[["supervisado", "identidad", "vigilante", "or", "invisibles", "n"]]
registrar("Evaluación E4C", "1,000", arp["nativo"]["supervisado"], "robustez_e4c.json", "resumen/recall_arp_por_paso/nativo/supervisado")
registrar("Evaluación E4C", "0,000", arp["5s"]["supervisado"], "robustez_e4c.json", "resumen/recall_arp_por_paso/5s/supervisado")
registrar("Evaluación E4C", "1,000", arp["5s"]["vigilante"], "robustez_e4c.json", "resumen/recall_arp_por_paso/5s/vigilante")
registrar("Evaluación E4C", "0,0", e4c["fa_vigilante_variacion_por_dispositivo_dia"], "robustez_e4c.json", "resumen/fa_vigilante_variacion_por_dispositivo_dia", 1)
print("FA del vigilante con variación benigna:", e4c["fa_vigilante_variacion_por_dispositivo_dia"], "FA/disp-día")
tabla

FA del vigilante con variación benigna: 0.0 FA/disp-día


,supervisado,identidad,vigilante,or,invisibles,n
nativo,1.0,0.5,1.0,1.0,0.0,2.0
5s,0.0,1.0,1.0,1.0,2.0,2.0
15s,0.0,1.0,1.0,1.0,2.0,2.0
30s,0.0,1.0,1.0,1.0,2.0,2.0
60s,0.0,1.0,1.0,1.0,2.0,2.0


### 5.3 Menos falsas alarmas sin absorber el clon (E5C)

Adopción por persistencia exclusiva: FA con variación benigna **826 → 382/disp-día** sin perder la
detección de clones de pila distinta. Precisión: `resumen` guarda 381,5 (un decimal) y `por_modo`
el valor completo, 381,46; «382» es el redondeo de 381,5 (doble redondeo: el valor completo da 381).

Fuente: `reports/adopcion_e5c.json` → `resumen` y `por_modo/persistencia/fa/con_variacion` (E5C).

In [10]:
a = leer("adopcion_e5c.json")
fa = a["resumen"]["fa_con_variacion"]
ic = a["por_modo"]["persistencia"]["fa"]["con_variacion"]["fa_ventanas_por_dispositivo_dia"]["ic95"]
print(f"FA con variación: apariciones {coma(fa['apariciones'], 1)} → persistencia {coma(fa['persistencia'], 1)} "
      f"FA/disp-día (IC95 [{coma(ic[0], 1)}, {coma(ic[1], 1)}])")
registrar("Evaluación E5C", "826", round(fa["apariciones"]), "adopcion_e5c.json", "resumen/fa_con_variacion/apariciones (redondeo)", 0)
registrar("Evaluación E5C", "382", round(fa["persistencia"]), "adopcion_e5c.json", "resumen/fa_con_variacion/persistencia (redondeo)", 0)
pd.DataFrame(a["resumen"]["deteccion_por_nivel"]).T

FA con variación: apariciones 826,2 → persistencia 381,5 FA/disp-día (IC95 [326,2, 442,0])


,apariciones,persistencia
ingenuo,0.975,0.975
pila_completa,0.200,0.200
ritmo,1.000,0.975
suplantacion_ip,0.950,0.950
victima_apagada,1.000,1.000


## 6. Despliegue — streaming, mitigación y pronóstico (OE3; E5, E6)

E5 y E6 no dejaron JSON en `reports/`: sus cifras viven en `docs/INFORME_E5.md` y
`docs/INFORME_E6.md`. La celda siguiente lo comprueba en vez de suponerlo, y después demuestra los
módulos de despliegue **sin red ni datos**: la mitigación corre en *dry-run* (registra el comando,
no lo ejecuta) y el pronóstico usa una serie de juguete. Esas salidas son demostración del módulo,
no cifras del informe.

In [11]:
CLAVES = {"E5": ("streaming", "captura"), "E6": ("mitigacion", "pronostico")}
INFORMES = {"E5": "docs/INFORME_E5.md", "E6": "docs/INFORME_E6.md"}
for etapa, claves in CLAVES.items():
    hay = [p.name for p in (RAIZ / "reports").glob("*.json") if any(c in p.name for c in claves)]
    print(f"{etapa}: {hay}" if hay else f"{etapa}: cifra no encontrada en reports/, ver informe {INFORMES[etapa]}")

from src.mitigacion import GestorMitigacion, PoliticaMitigacion
from src.pronostico import PronosticadorAnomalias
import numpy as np

g = GestorMitigacion(politica=PoliticaMitigacion(umbral_bloqueo=3, ventana_s=60))
for i in range(4):
    nivel = g.procesar_alerta("02:ff:00:00:00:99", ts_s=float(i), score=0.9)
print("mitigación · modo:", g.modo, "| último nivel:", nivel.name,
      "| acciones:", [x["accion"] for x in g.acciones])
p = PronosticadorAnomalias(metodo="holt").entrenar(np.array([0, 0, 1, 1, 2, 3, 4, 6, 8], dtype=float))
print("pronóstico Holt (serie de juguete) · tendencia:", p.tendencia, "| aviso:", p.aviso_temprano(umbral=10))

[dry-run] bloqueo de 02:ff:00:00:00:99 → ebtables -A FORWARD -s 02:ff:00:00:00:99 -j DROP


E5: cifra no encontrada en reports/, ver informe docs/INFORME_E5.md
E6: cifra no encontrada en reports/, ver informe docs/INFORME_E6.md


mitigación · modo: dry_run | último nivel: BLOQUEO | acciones: ['alerta', 'bloqueo']
pronóstico Holt (serie de juguete) · tendencia: creciente | aviso: {'aviso': True, 'tendencia': 'creciente', 'pronostico': [8.264, 9.514, 10.763, 12.012, 13.262], 'umbral': 10}


## Verificación: cifras del texto contra `reports/`

Cada cifra citada arriba, con el valor que trae el JSON y si coincide al redondeo del texto.

In [12]:
ver = pd.DataFrame(VERIFICACION)
print(f"{int(ver['coincide'].sum())} de {len(ver)} cifras coinciden con reports/; "
      f"tiempo total del cuaderno: {time.perf_counter() - T_INICIO:.2f} s")
ver

23 de 23 cifras coinciden con reports/; tiempo total del cuaderno: 0.30 s


,fase,cifra del texto,valor en JSON,archivo,ruta,coincide
0,Preparación,52,52.000000,reports/evaluacion_e2.json,columnas_modelo (longitud),True
1,Modelado E2,"0,894",0.893617,reports/evaluacion_e2.json,modelos/xgboost/resumen/episodios/recall/valor,True
2,Modelado E2,"0,856",0.856250,reports/evaluacion_e2.json,modelos/xgboost/resumen/episodios/recall/ic95/0,True
3,Modelado E2,"0,936",0.936170,reports/evaluacion_e2.json,modelos/xgboost/resumen/episodios/recall/ic95/1,True
4,Modelado E2,"0,89",0.893909,reports/evaluacion_e2.json,modelos/xgboost/resumen/dispositivos/falsas_al...,True
5,Modelado LOAO,"0,228",0.228000,reports/extractor_final.json,vigente/xgboost/media/valor,True
6,Modelado LOAO,"0,168",0.168000,reports/extractor_final.json,vigente/xgboost/media/ic95/0,True
7,Modelado LOAO,"0,286",0.286000,reports/extractor_final.json,vigente/xgboost/media/ic95/1,True
8,Modelado Anomalías,"0,185",0.185106,reports/anomalias.json,resultados/«isolation_forest/sin_capa2»/resume...,True
9,Modelado E3,"16,19",16.190000,reports/identidad_e3b.json,resumen/ipid/fa_antes,True


## Conclusiones

1. El supervisado detecta el catálogo conocido (**0,894 [0,856, 0,936]** a **0,89 FA/disp-día**,
   E2), pero no transfiere a tipos nuevos (**LOAO 0,228**); un detector solo-benigno tampoco (0,185).
2. El perfil de pila detecta clones de pila distinta en segundos a **0,38 FA/disp-día** (E3B); se
   rompe con el clon de pila completa (0,20) y con variación benigna realista sube a 826 FA/disp-día.
3. Robustez (E4R): el recall no baja de 0,5 a ninguna intensidad (media 0,935, suave 0,891,
   ráfaga 0,824) y ningún episodio es invisible por construcción.
4. Evasión (E4C): el ARP lento es invisible para el supervisado (**1,000 → 0,000**) pero el
   vigilante IP→MAC lo detecta al **1,000** con 0,0 FA/disp-día (n = 2 episodios por paso).
5. E5C baja las FA con variación de 826 a 382/disp-día sin perder detección; E5/E6 dejan el camino
   en vivo, la mitigación en dry-run y el pronóstico implementados (cifras en sus informes).

Límites: todo es laboratorio sintético con sesiones de 15 min; la captura real, DHCP/failover, los
RST del clon en un switch y las sesiones largas (≥ 30 nuevas) quedan para el laboratorio físico.